# Use-case: Golf Forecasting

Generate a forecasting dataset about professional golf (tournaments, majors, rankings) using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments.

In [1]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

TEST_START = "2025-08-01"

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for golf forecasting.

In [3]:
instructions = """
Generate binary forecasting questions about professional golf across all major tours and events.

Cover what golf fans bet on: tournament outcomes, cuts, matchups, majors, team events, season races, world rankings, and player milestones.

Questions should be specific, verifiable, and span the full probability spectrum.
"""

good_examples = [
    "Will Scottie Scheffler win the 2025 Masters?",
    "Will the 2025 US Open winning score be under par?",
    "Will Tiger Woods make the cut at the 2025 Masters?",
    "Will Rory McIlroy finish top 5 at the 2025 US Open?",
    "Will any LIV player win a major championship in 2025?",
    "Will Europe win the 2025 Ryder Cup?",
    "Will any player win 4+ PGA Tour events in 2025?",
    "Will Scottie Scheffler remain world #1 through June 2025?",
    "Will a first-time major winner emerge at the 2025 PGA Championship?",
    "Will Nelly Korda win the 2025 US Women's Open?",
]

bad_examples = [
    "Will someone win the tournament? (obvious)",
    "Will golf be exciting? (subjective)",
    "Will there be birdies? (trivial)",
]

search_queries = [
    "PGA Tour",
    "LIV Golf",
    "LPGA",
    "golf major championship",
    "Ryder Cup Presidents Cup",
    "golf world rankings",
    "professional golf",
    "women's golf",
    "European Tour golf",
]

In [4]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2024, 6, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=14,
        search_query=search_queries,
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=3,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [5]:
dataset = lr.transforms.run(pipeline, max_questions=500, name="Golf forecasting")

samples = dataset.download()
pct = (dataset.valid_count() / len(samples) * 100) if samples else 0
print(f"Generated {len(samples)} samples ({pct:.1f}% valid)")


Output()

Generated 450 samples (78.7% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [10]:
from lightningrod.utils import deduplicate_samples, test_train_split, filter_samples, add_rl_training_fields, render_sample, flatten_samples

valid_samples = filter_samples(samples, drop_missing_context=False)
samples_prep = deduplicate_samples(valid_samples)
print(f"Valid+deduped: {len(samples_prep)}")

train, test = test_train_split(samples_prep, test_fraction=0.2)

def _yes_count(sample_list):
    return sum(
        1 for s in sample_list
        if s.label and str(s.label.label).lower() in ("1", "true", "yes")
    )

for name, data in [("Train", train), ("Test", test)]:
    yes = _yes_count(data)
    print(f"{name}: {len(data)} rows, {yes/len(data)*100:.1f}% yes")

add_rl_training_fields(train, answer_type)
_ = add_rl_training_fields(test, answer_type)

Valid+deduped: 352
Train: 156 rows, 37.8% yes
Test: 71 rows, 43.7% yes


In [11]:
def _display_head(data, name, n=5):
    rows = flatten_samples(data[:n])
    if not rows:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(rows)
    cols = [
        "question_text", "label", "correct_answer",
        "question_prediction_date", "question_date_close", "label_resolution_date",
        "answer_type", "answer_parser_type", "reward_function_type",
        "label_confidence",
    ]
    display_cols = [c for c in cols if c in df.columns]
    rename = {
        "question_text": "Question",
        "label": "Answer",
        "label_confidence": "Confidence",
        "correct_answer": "Correct",
        "question_prediction_date": "Prediction Date",
        "question_date_close": "Close Date",
        "label_resolution_date": "Resolution Date",
        "answer_type": "Type",
        "answer_parser_type": "Parser",
        "reward_function_type": "Reward",
    }
    print(f"{name} (head):")
    display(df[display_cols].rename(columns=rename))

_display_head(train, "Train")
_display_head(test, "Test")

print("Sample prompt (first train example):")
render_sample(train[0], answer_type=answer_type)[0]["content"]

Train (head):


,Question,Answer,Correct,Prediction Date,Close Date,Resolution Date,Type,Parser,Reward,Confidence
0,Will at least one Arizona Women's Golf student...,0,0.0,2024-07-15T00:00:00,2025-06-05T00:00:00,2025-05-27T00:00:00,binary,binary,binary_log_score,1.0
1,Will the University of Arizona Women's Golf te...,1,1.0,2024-07-15T00:00:00,2025-04-20T00:00:00,2025-04-17T00:00:00,binary,binary,binary_log_score,1.0
2,Will the University of Arizona Women's Golf te...,0,0.0,2024-07-15T00:00:00,2025-05-15T00:00:00,2025-05-06T00:00:00,binary,binary,binary_log_score,1.0
3,Will the Oakland University women's golf team ...,1,1.0,2024-07-16T00:00:00,2025-05-01T00:00:00,2025-04-21T00:00:00,binary,binary,binary_log_score,1.0
4,Will the Oakland University women's golf team ...,1,1.0,2024-07-16T00:00:00,2025-05-01T00:00:00,2025-04-21T00:00:00,binary,binary,binary_log_score,1.0


Test (head):


,Question,Answer,Correct,Prediction Date,Close Date,Resolution Date,Type,Parser,Reward,Confidence
0,Will Viktor Hovland win a major championship i...,0,0.0,2025-06-16T00:00:00,2025-07-21T00:00:00,2025-07-21T00:00:00,binary,binary,binary_log_score,1.0
1,Will Rory McIlroy win the 2025 Open Championship?,0,0.0,2025-06-16T00:00:00,2025-07-20T00:00:00,2025-07-20T00:00:00,binary,binary,binary_log_score,1.0
2,Will Bud Cauley finish in the top 10 of any PG...,0,0.0,2025-06-16T00:00:00,2025-09-01T00:00:00,2025-08-31T00:00:00,binary,binary,binary_log_score,0.9
3,Will J.J. Spaun be ranked in the top 5 of the ...,0,0.0,2025-06-16T00:00:00,2025-07-21T00:00:00,2025-07-21T00:00:00,binary,binary,binary_log_score,1.0
4,Will JJ Spaun win another PGA Tour event durin...,0,0.0,2025-06-16T00:00:00,2025-12-31T00:00:00,2025-12-31T00:00:00,binary,binary,binary_log_score,1.0


Sample prompt (first train example):


"QUESTION:\nWill at least one Arizona Women's Golf student-athlete be named a WGCA First-Team All-American for the 2024-25 season?\n\nTODAY'S DATE:\n2024-07-15\n\nRESOLUTION CRITERIA:\nThe question resolves to 'Yes' if the Women's Golf Coaches Association (WGCA) names any player current rostered for the University of Arizona to their First-Team All-American list following the 2025 NCAA Championships. These awards are typically announced in late May or early June.\n\nCLOSE DATE:\n2025-06-05\n\nCONTEXT:\nNEWS:\nRecent news articles relevant to this question:\n\n---\nARTICLES\n[1] 2024 Spring Scholar-Athletes of the Year Announced (published on 2024-07-03 by big12sports.com) (relevance: 1.0)\nSummary: The Big 12 Conference announced the 2024 Spring Scholar-Athletes of the Year. Winners were selected by sport head coaches based on academic achievement and athletic performance. Honorees include: Zach Ehrhard (Oklahoma State Baseball), Johnny Keefer (Baylor Men's Golf), Haley Vargas (Kansas 

## Model Training Results

We fine-tuned a forecasting model via RL on a dataset of 3,178 forecasting questions, surpassing GPT-5 performance.

<br>

![Brier Skill Score](https://huggingface.co/datasets/LightningRodLabs/GolfForecasting/resolve/main/brier_skill_score.png)

<br>

**For more details on methods, results, and data, visit the HuggingFace links below:**
- **[Golf-Forecaster Model](https://huggingface.co/LightningRodLabs/Golf-Forecaster)**
- **[Golf-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/GolfForecasting)**

<br>

---

<br>

🚀 **Coming Soon:** Seamlessly generate datasets, fine-tune, and evaluate your own forecasting models end-to-end on the Lightningrod platform.
 
👉 [Sign up to get early access and updates.](https://lightningrod.ai/)